# RAVE Dissertation Experiments
**Mihir Apte — MSc Data Science, TCD**

Before running:
1. **Runtime > Change runtime type > A100 GPU**
2. Run cells top to bottom
3. Upload your 4 videos when Cell 6 prompts you

**Methods tested:**
- Baseline (random shuffle, depth_zoe)
- Semantic v1 (greedy NN, depth_zoe)
- Semantic v2 (K-means, depth_zoe)
- Multi-ControlNet (random shuffle, depth_zoe + canny + FreeU)


In [ ]:
# Cell 1 - Check GPU
import torch
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU detected. Switch runtime to A100 before continuing.")


In [ ]:
# Cell 2 - Clone or update repo
import os
REPO = "/content/dissertation-mihir"
if os.path.exists(REPO):
    print("Repo exists — pulling latest changes...")
    !cd {REPO} && git pull origin main
else:
    print("Cloning repo...")
    !git clone https://github.com/MihirApte/dissertation-mihir.git {REPO}
print("Done.")


In [ ]:
# Cell 3 - Install packages
# torch/torchvision already installed by Colab - do NOT reinstall
import subprocess
pkgs = [
    "diffusers",
    "transformers",
    "accelerate",
    "xformers",
    "omegaconf",
    "einops",
    "open_clip_torch",
    "scikit-learn",
    "Pillow",
    "opencv-python",
    "imageio",
    "imageio-ffmpeg",
]
subprocess.run(["pip", "install", "-q"] + pkgs, check=True)
subprocess.run(["pip", "install", "-q",
    "git+https://github.com/openai/CLIP.git"], check=True)
print("All packages installed.")


In [ ]:
# Cell 4 - Patch basicsr (torchvision removed functional_tensor in 0.17+)
import glob
matches = glob.glob("/usr/local/lib/python3.*/dist-packages/basicsr/data/degradations.py")
if matches:
    path = matches[0]
    with open(path) as f:
        content = f.read()
    old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    new = "from torchvision.transforms.functional import rgb_to_grayscale"
    if old in content:
        with open(path, "w") as f:
            f.write(content.replace(old, new))
        print("basicsr patched")
    else:
        print("basicsr already patched")
else:
    print("basicsr not found - skipping patch")


In [ ]:
# Cell 5 - Verify ZoeDepth uses GPU (A100 has 80GB - no CPU workaround needed)
import torch
zoe_path = "/content/dissertation-mihir/annotator/zoe/__init__.py"
with open(zoe_path) as f:
    content = f.read()
# Make sure it is NOT patched to CPU (revert if it was)
if 'self.model.to("cpu")' in content:
    content = content.replace('self.model.to("cpu")', "self.model.to(self.device)")
    content = content.replace('torch.from_numpy(image_depth).float().to("cpu")',
                             "torch.from_numpy(image_depth).float().to(self.device)")
    with open(zoe_path, "w") as f:
        f.write(content)
    print("ZoeDepth reverted to GPU (correct for A100)")
else:
    print("ZoeDepth already using GPU - no change needed")


In [ ]:
# Cell 6 - Upload videos
# Run this cell once per video. Upload all 4: truck.mp4, shanghai.mp4, street.mp4, dog.mp4
import os, shutil
from google.colab import files

VIDEO_DIR = "/content/dissertation-mihir/data/mp4_videos"
os.makedirs(VIDEO_DIR, exist_ok=True)

VIDEOS_NEEDED = ["truck.mp4", "shanghai.mp4", "street.mp4", "dog.mp4"]
missing = [v for v in VIDEOS_NEEDED if not os.path.exists(os.path.join(VIDEO_DIR, v))]

if not missing:
    print("All 4 videos already present:")
    for v in VIDEOS_NEEDED:
        size = os.path.getsize(os.path.join(VIDEO_DIR, v)) / 1e6
        print(f"  {v}  ({size:.1f} MB)")
else:
    print(f"Missing videos: {missing}")
    print("Select ALL missing video files at once in the upload dialog...")
    uploaded = files.upload()
    for fname in uploaded:
        dest = os.path.join(VIDEO_DIR, os.path.basename(fname))
        shutil.move(fname, dest)
        print(f"Saved: {dest}")
    missing_after = [v for v in VIDEOS_NEEDED if not os.path.exists(os.path.join(VIDEO_DIR, v))]
    if missing_after:
        print(f"Still missing: {missing_after} - re-run this cell to upload them.")
    else:
        print("All videos ready!")


In [ ]:
# Cell 7 - Environment check
import os
os.chdir("/content/dissertation-mihir")
!python3 check_gpu.py


In [ ]:
# Cell 8 - Run ALL experiments
# This runs baseline, semantic v1 (greedy), semantic v2 (kmeans),
# and multi-controlnet for each available video.
# Expected total time: 3-6 hours on A100 for all 4 videos x 4 methods
# The script skips any video whose .mp4 is not in data/mp4_videos/
import os
os.chdir("/content/dissertation-mihir")
!bash run_all_experiments.sh 2>&1


In [ ]:
# Cell 9 - Show full metrics table
import os
os.chdir("/content/dissertation-mihir")
results_file = "results/metrics_all_methods.txt"
if os.path.exists(results_file):
    with open(results_file) as f:
        print(f.read())
else:
    print("metrics_all_methods.txt not found.")
    print("Running compute_metrics_all.py now...")
    !python3 compute_metrics_all.py --device cuda


In [ ]:
# Cell 10 - Download all GIFs and metrics as a zip
import shutil
from google.colab import files
shutil.make_archive("/content/rave_results", "zip", "/content/dissertation-mihir/results")
files.download("/content/rave_results.zip")
print("Downloading rave_results.zip...")
print("Extract on Windows with 7-zip or rename files to shorter names if path-too-long error appears.")
